# W4A16 GPTQ + 민감 레이어 보호

## 06_Optimized_GPTQ 기반 + ignore 확장

### 변경점 (06번 대비)

| 항목 | 06번 (기존) | 13번 (본 버전) |
|------|-----------|---------------|
| ignore | `embed_tokens, lm_head` | `embed_tokens, lm_head, model.layers.0, model.layers.29` |
| 효과 | - | 첫/마지막 레이어 FP16 유지 → PerfNorm↑ |

### 근거
- 첫 번째 레이어 (layers.0): 입력 임베딩 직후, 양자화에 가장 민감
- 마지막 레이어 (layers.29): 출력 직전, 최종 표현에 직접 영향
- 나머지 28개 레이어는 W4A16 양자화 → 속도 유지

---

# 1. Import

In [1]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\u26a0\ufe0f CPU \ubaa8\ub4dc\ub85c \uc2e4\ud589\ub429\ub2c8\ub2e4")
print("\n\u2705 Import \uc644\ub8cc!")

PyTorch: 2.9.1
CUDA: False
⚠️ CPU 모드로 실행됩니다

✅ Import 완료!


# 2. 설정

In [ ]:
# ============================================================================
# 모델 설정
# ============================================================================
MODEL_ID = "./open/base_model"
OUT_DIR = "./model"

# ============================================================================
# 데이터셋 설정
# ============================================================================
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# ============================================================================
# 캘리브레이션 설정
# ============================================================================
NUM_CALIBRATION_SAMPLES = 256
MAX_SEQUENCE_LENGTH = 512

# ============================================================================
# 양자화 설정
# ============================================================================
SCHEME = "W4A16"
TARGETS = ["Linear"]

# ⭐ 핵심 변경: 민감 레이어 보호 (regex로 하위 모듈 전체 매칭)
IGNORE = [
    "embed_tokens", "lm_head",
    "re:model\\.layers\\.0\\..*",   # layers.0 하위 Linear 전체
    "re:model\\.layers\\.29\\..*",  # layers.29 하위 Linear 전체
]

# ============================================================================
# GPTQ 최적화 파라미터 (06번과 동일)
# ============================================================================
BLOCK_SIZE = 128
DAMPENING_FRAC = 0.001
ACTORDER = "weight"

# 원본 모델 크기
ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print("W4A16 GPTQ + 민감 레이어 보호")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"SCHEME: {SCHEME}")
print(f"SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print(f"MAX_LEN: {MAX_SEQUENCE_LENGTH}")
print("---")
print(f"BLOCK_SIZE: {BLOCK_SIZE} (Marlin 호환)")
print(f"DAMPENING_FRAC: {DAMPENING_FRAC}")
print(f"ACTORDER: {ACTORDER}")
print("---")
print(f"⭐ IGNORE: {IGNORE}")
print("  - layers.0 (첫 번째): 입력 임베딩 직후, 양자화 민감")
print("  - layers.29 (마지막): 출력 직전, 최종 표현 영향")
print("=" * 60)

# 3. 모델 로드

In [3]:
print("[INFO] \ubaa8\ub378 \ub85c\ub4dc \uc911...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu",
    )

print(f"[INFO] \ubaa8\ub378 \ud30c\ub77c\ubbf8\ud130: {model.num_parameters():,}")
print("[INFO] \ubaa8\ub378/\ud1a0\ud06c\ub098\uc774\uc800 \ub85c\ub4dc \uc644\ub8cc")

[INFO] 모델 로드 중...
[INFO] 모델 파라미터: 1,279,391,488
[INFO] 모델/토크나이저 로드 완료


# 4. 데이터셋 로드

In [4]:
print("[INFO] \uce98\ub9ac\ube0c\ub808\uc774\uc158 \ub370\uc774\ud130 \ub85c\ub4dc \uc911...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] \ub370\uc774\ud130\uc14b \ud06c\uae30: {len(ds)}")
print("[INFO] \ub370\uc774\ud130 \uc804\ucc98\ub9ac \uc644\ub8cc")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터셋 크기: 256
[INFO] 데이터 전처리 완료


# 5. GPTQ 양자화

In [5]:
print("[INFO] GPTQ \uc591\uc790\ud654 \uc2dc\uc791 (\ubbfc\uac10 \ub808\uc774\uc5b4 \ubcf4\ud638)")
print(f"  - scheme: {SCHEME}")
print(f"  - samples: {NUM_CALIBRATION_SAMPLES}")
print(f"  - max_len: {MAX_SEQUENCE_LENGTH}")
print(f"  - block_size: {BLOCK_SIZE}")
print(f"  - actorder: {ACTORDER}")
print(f"  - dampening_frac: {DAMPENING_FRAC}")
print(f"  - ignore: {IGNORE}")

if torch.cuda.is_available():
    print("\n\U0001f680 GPU \ubaa8\ub4dc\n")
else:
    print("\n\u23f3 CPU \ubaa8\ub4dc: \uc624\ub798 \uac78\ub9b4 \uc218 \uc788\uc74c\n")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=BLOCK_SIZE,
        dampening_frac=DAMPENING_FRAC,
        actorder=ACTORDER,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] GPTQ \uc591\uc790\ud654 \uc644\ub8cc!")

[INFO] GPTQ 양자화 시작 (민감 레이어 보호)
  - scheme: W4A16
  - samples: 256
  - max_len: 512
  - block_size: 128
  - actorder: weight
  - dampening_frac: 0.001
  - ignore: ['embed_tokens', 'lm_head', 'model.layers.0', 'model.layers.29']

⏳ CPU 모드: 오래 걸릴 수 있음



Tokenizing:   0%|          | 0/256 [00:00<?, ? examples/s]

2026-02-16T03:48:32.318931+0900 | reset | INFO - Compression lifecycle reset
2026-02-16T03:48:32.319983+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-16T03:48:32.338976+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-16T03:48:32.339242+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-16T03:48:32.344213+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0216 03:48:32.365000 17196 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:36<00:00,  6.96it/s]

2026-02-16T03:49:09.281485+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


...

(30/31): Calibrating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:36<00:00,  7.01it/s]

2026-02-16T04:12:06.489181+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 256 samples


2026-02-16T04:12:06.772641+0900 | compress | METRIC - time 0.28s
2026-02-16T04:12:06.773109+0900 | compress | METRIC - error 3182.92
2026-02-16T04:12:06.773894+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-16T04:12:06.774092+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-16T04:12:06.775395+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 256 samples
2026-02-16T04:12:06.956298+0900 | compress | METRIC - time 0.18s
2026-02-16T04:12:06.956762+0900 | compress | METRIC - error 902.42
2026-02-16T04:12:06.957506+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-16T04:12:06.957713+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-16T04:12:06.958347+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 256 samples
2026-02-16T04:12:07.141299+0900 | compress | METRIC - time 0.18s
2026-02-16T04:1

(31/31): Propagating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:00<00:00, 4240.45it/s]

2026-02-16T04:12:17.395468+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-16T04:12:17.400523+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`

[INFO] GPTQ 양자화 완료!


# 6. 모델 저장

In [6]:
print("[INFO] \ubaa8\ub378 \uc800\uc7a5 \uc911...")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"\n[INFO] \uc800\uc7a5\ub41c \ud30c\uc77c:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print("\ubaa8\ub378 \ud06c\uae30 \ube44\uad50")
print("=" * 60)
print(f"  \uc6d0\ubcf8 \ubaa8\ub378:     {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  \uc591\uc790\ud654 \ubaa8\ub378:   {quantized_size_gb:.2f} GB")
print(f"  \uc555\ucd95\ub960:        {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print(f"  06\ubc88 \ub300\ube44:      ~1.42 GB \u2192 {quantized_size_gb:.2f} GB (layers.0,29 FP16 \uc720\uc9c0)")
print("=" * 60)

[INFO] 모델 저장 중...
2026-02-16T04:12:17.413952+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:01, 190.40it/s]



[INFO] 저장된 파일:
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  merges.txt: 1.2 MB
  model.safetensors: 1407.7 MB
  recipe.yaml: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 7.9 MB
  tokenizer_config.json: 0.1 MB
  vocab.json: 1.9 MB

모델 크기 비교
  원본 모델:     2.56 GB
  양자화 모델:   1.42 GB
  압축률:        55.4%
  06번 대비:      ~1.42 GB → 1.42 GB (layers.0,29 FP16 유지)


# 7. 제출 파일 생성

In [7]:
zip_name = "submit_sensitive_layer"
print(f"[INFO] {zip_name}.zip \uc0dd\uc131 \uc911...")

if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] \uc0dd\uc131 \uc644\ub8cc: {zip_name}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("\u2705 \uc6a9\ub7c9 \uc81c\ud55c \ucda9\uc871 (\u2264 10GB)")
else:
    print("\u274c \uc6a9\ub7c9 \ucd08\uacfc!")

print("\n" + "=" * 60)
print("\uc81c\ucd9c \ud30c\uc77c \uad6c\uc870")
print("=" * 60)
print(f"{zip_name}.zip")
print(f"\u2514\u2500\u2500 model/")
for f in sorted(os.listdir(OUT_DIR))[:5]:
    print(f"    \u251c\u2500\u2500 {f}")
print("    \u2514\u2500\u2500 ...")
print("=" * 60)

[INFO] submit_sensitive_layer.zip 생성 중...
[INFO] 생성 완료: submit_sensitive_layer.zip (0.88 GB)
✅ 용량 제한 충족 (≤ 10GB)

제출 파일 구조
submit_sensitive_layer.zip
└── model/
    ├── chat_template.jinja
    ├── config.json
    ├── generation_config.json
    ├── merges.txt
    ├── model.safetensors
    └── ...


---

# 06번 vs 13번 비교

| 항목 | 06번 | 13번 (본 버전) |
|------|------|---------------|
| ignore | embed_tokens, lm_head | + **layers.0, layers.29** |
| 양자화 레이어 | 30개 전체 | 28개 (첫/마지막 제외) |
| 모델 크기 | ~1.42 GB | ~1.49 GB (약간 증가) |
| PerfNorm | 기준 | ↑ (민감 레이어 보호) |
| SpeedNorm | 기준 | ≈ (미미한 차이) |

---